# Motion tracking people in streat 

In [2]:
import cv2
import numpy as np
import os
os.chdir(r"C:\Users\DT1\Desktop\proj-python\Videos")

In [7]:
cap=cv2.VideoCapture("motion_walking.mp4")
# cap=cv2.VideoCapture(0)
# cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
# cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
h=cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
w=cap.get(cv2.CAP_PROP_FRAME_WIDTH)
print("Height:", h, "Width:", w)
fourcc=cv2.VideoWriter_fourcc(*'XVID')
out=cv2.VideoWriter('output.avi', fourcc, 30.0, (int(w), int(h)))

print("FPS", cap.get(cv2.CAP_PROP_FPS))
 
ret,frame_1=cap.read()
ret,frame_2=cap.read()

while cap.isOpened():
    diff=cv2.absdiff(frame_1, frame_2)
    gray=cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
    blur=cv2.GaussianBlur(gray, (5,5), 0)
    _, thresh=cv2.threshold(blur, 20, 255, cv2.THRESH_BINARY)
    dilated=cv2.dilate(thresh, None, iterations=3)
    # erode=cv2.erode(dilated,None,iterations=3)
    contours, _=cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    draw=cv2.drawContours(frame_1, contours, -1, (100,55,20), 1)
    for contour in contours:
        (x,y,w,h)=cv2.boundingRect(contour)
        if cv2.contourArea(contour)<1250:
            continue
        cv2.rectangle(frame_1, (x,y), (x+w, y+h), (0,255,0), 3)
        cv2.putText(frame_1, "Movement Detected", (15,40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)
    cv2.imshow("frame", frame_1)
    out.write(frame_1)
    frame_1=frame_2
    ret, frame_2=cap.read()
    if cv2.waitKey(40)==ord('q'):
        break
cap.release()
cv2.destroyAllWindows()

Height: 360.0 Width: 640.0
FPS 29.97002997002997


# Tracking trafic cars in rightways

In [ ]:
# !pip install --upgrade pip
# !pip install tracker

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----------- ---------------------------- 0.5/1.8 MB 1.4 MB/s eta 0:00:01
   ----------------- ---------------------- 0.8/1.8 MB 1.2 MB/s eta 0:00:01
   ----------------------- ---------------- 1.0/1.8 MB 1.2 MB/s eta 0:00:01
   ---------------------------- ----------- 1.3/1.8 MB 1.1 MB/s eta 0:00:01
   ---------------------------------- ----- 1.6/1.8 MB 1.2 MB/s eta 0:00:01
   ---------------------------------- ----- 1.6/1.8 MB 1.2 MB/s eta 0:00:01
   ---------------------------------- ----- 1.6/1.8 MB 1.2 MB/s eta 0:00:01
   ---------------------------------- ----- 1.6/1.8 MB 1.2 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 836.1 kB/s  0:00:02


ERROR: To modify pip, please run the following command:
C:\Users\DT1\Desktop\proj-python\.venv\Scripts\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [49]:
import cv2
import numpy as np
from tracker import *
import os
os.chdir(r"C:\Users\DT1\Desktop\proj-python\Videos")

cap = cv2.VideoCapture("cars_motion.mp4")
tracker=EuclideanDistTracker()

object_detector = cv2.createBackgroundSubtractorMOG2(history=100, varThreshold=40)

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # Detect objects
    mask = object_detector.apply(frame)
    # roi=frame[100:720,200:600]
    # Optional: morphological operations to clean up mask
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    # mask=cv2.adaptiveThreshold(mask, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)
    
    # Find contours
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    # cv2.drawContours(frame, contours, -1, (0, 55, 0), 2)
    # Draw bounding boxes around detected objects
    detection=[]
    for contour in contours:
        area = cv2.contourArea(contour)
        if area > 5000:  # Filter small noise
            x, y, w, h = cv2.boundingRect(contour)
            detection.append([x, y, w, h])
         
    objects=tracker.update(np.array(detection))
    for obj in objects:
        x,y,w,h,id=obj
        cv2.putText(frame, str(id), (x, y - 10), cv2.FONT_HERSHEY_PLAIN, 1, (0, 255, 0), 2)
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(frame, "Vehicle", (x, y - 30), cv2.FONT_HERSHEY_PLAIN, 1, (0, 255, 0), 2)
    cv2.imshow("Frame", frame)
    # cv2.imshow("ROI",roi)
    cv2.imshow("Mask", mask)
    
    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()